# Hourly preprocessing: January 2024

Count pickups on the complete hourly grid for taxi zones 1–263, including zero-demand hours.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_01_cleaned.parquet"))


In [ ]:
df_copy = df.copy()


In [ ]:
df_copy


In [ ]:
def groupby_hour(df):
    df_hourly = df.groupby([pd.Grouper(key = "tpep_pickup_datetime", freq = "h"), "PULocationID"]).size().reset_index(name = "trip_count")

    hours = pd.date_range(
        start = "2024-01-01",
        end = "2024-02-01",
        freq = "h",
        inclusive = "left"
    )

    zones = range(1, 264)

    full_index = pd.MultiIndex.from_product(
        [hours, zones],
        names = ["tpep_pickup_datetime", "PULocationID"])

    df_hourly = (
    df_hourly
    .set_index(["tpep_pickup_datetime", "PULocationID"])
    .reindex(full_index, fill_value = 0)
    .reset_index())

    return df_hourly

df_hourly = groupby_hour(df_copy)


In [ ]:
df_hourly.to_parquet(
    str(project_root / "data/yellow_taxi_2024_01_aggreg.parquet"),
    index = False
)
